In [48]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pyarrow.feather as feather

import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')


# Dataset Definition

In [49]:
seasonal_and_smooth = True 
dataset_imputed= False 
data_andre = False



In [50]:
path = ''
if seasonal_and_smooth:
    path = '../dataset/seasonal_and_smooth.feather'
elif dataset_imputed:
    path = '../dataset/df_imputed.feather'
elif data_andre:
    path = '../dataset/data_andre.feather'

In [51]:
table = feather.read_table(path, memory_map=True)
df_selected = table.to_pandas()
df_selected.head()



,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,promo_value_GAS,...,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id,item_label
date,,,,,,,,,,,,,,,,,,,,,
2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,seasonal
2021-01-23,952568,6,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,1,0.0,6269,seasonal
2021-01-23,809,13,juices drnks shelf stbl,pos subd grocery other,pos dept grocery,juice/aseptic/new age,0,0.0,1,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,seasonal
2021-01-23,20405,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,seasonal
2021-01-23,605573,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,6269,seasonal


In [52]:
# Check the dataframe structure first
print("Columns in df_selected:", df_selected.columns.tolist())
print("\nFirst few rows:")
print(df_selected.head())
print("\nData types:")
print(df_selected.dtypes)

# If 'date' is in the index, reset it
if 'date' not in df_selected.columns:
    if df_selected.index.name == 'date' or isinstance(df_selected.index, pd.DatetimeIndex):
        df_selected = df_selected.reset_index()
        print("\n✓ Date column recovered from index")

# Ensure date is datetime
if 'date' in df_selected.columns:
    df_selected['date'] = pd.to_datetime(df_selected['date'])
    print(f"✓ Date column is now datetime: {df_selected['date'].dtype}")
else:
    print("⚠️ Warning: 'date' column still missing!")

Columns in df_selected: ['item_id', 'value', 'cat_label', 'sdep_label', 'dep_label', 'dmn_label', 'promo_type_FRPG', 'promo_value_FRPG', 'promo_type_GAS', 'promo_value_GAS', 'promo_type_BOGO', 'promo_value_BOGO', 'promo_type_DISC', 'promo_value_DISC', 'promo_type_CIRC', 'promo_value_CIRC', 'promo_type_CIRE', 'promo_value_CIRE', 'promo_type_CLCP', 'promo_value_CLCP', 'promo_type_LFPE', 'promo_value_LFPE', 'store_id', 'item_label']

First few rows:
            item_id  value                cat_label              sdep_label  \
date                                                                          
2021-01-23       27      8       refrigerated drnks    pos subd dairy other   
2021-01-23   952568      6     carbonated sft drnks  pos subd grocery other   
2021-01-23      809     13  juices drnks shelf stbl  pos subd grocery other   
2021-01-23    20405      3                dairy chs    pos subd dairy other   
2021-01-23   605573      5       refrigerated drnks    pos subd dairy other

In [53]:
TRAIN_RATIO = 0.6      # 60% train (and lookback)
VAL_RATIO = 0.2        # 20% validation
TEST_RATIO = 0.2       # 20% test (forecast horizon)
BATCH_SIZE = 32
HIDDEN_SIZE = 128
NUM_LAYERS = 2
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

Using device: cpu


In [54]:
class TimeSeriesDataset(Dataset):
    """Dataset for LSTM time series forecasting."""
    
    def __init__(self, data: pd.DataFrame, lookback: int, forecast_horizon: int, 
                 store_id: int = None, item_id: int = None,
                 include_features: List[str] = None):
        """
        Parameters
        ----------
        data : pd.DataFrame
            DataFrame with columns: date, store_id, item_id, value, and optional features
        lookback : int
            Number of past days to use as input
        forecast_horizon : int
            Number of days to forecast
        store_id, item_id : int, optional
            If provided, filter to specific store-item pair
        include_features : List[str], optional
            Additional feature columns to include (e.g., promo columns)
        """
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        
        # Filter data if store/item specified
        if store_id is not None and item_id is not None:
            data = data[(data['store_id'] == store_id) & (data['item_id'] == item_id)].copy()
        
        # Sort by date
        data = data.sort_values('date').reset_index(drop=True)
        
        # Extract target (sales values)
        self.values = data['value'].values
        
        # Extract additional features if specified
        self.features = None
        if include_features:
            feature_cols = [col for col in include_features if col in data.columns]
            if feature_cols:
                self.features = data[feature_cols].values
        
        # Create sequences
        self.X, self.y = self._create_sequences()
        
    def _create_sequences(self):
        """Create input-output sequence pairs."""
        X, y = [], []
        
        for i in range(len(self.values) - self.lookback - self.forecast_horizon + 1):
            # Input: lookback window
            x_seq = self.values[i:i + self.lookback]
            
            # Include features if available
            if self.features is not None:
                x_features = self.features[i:i + self.lookback]
                x_seq = np.column_stack([x_seq, x_features])
            else:
                x_seq = x_seq.reshape(-1, 1)
            
            # Output: forecast horizon
            y_seq = self.values[i + self.lookback:i + self.lookback + self.forecast_horizon]
            
            X.append(x_seq)
            y.append(y_seq)
        
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.X[idx]), torch.FloatTensor(self.y[idx])







In [55]:
class LSTMForecaster(nn.Module):
    """LSTM model for multi-step forecasting."""
    
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, 
                 forecast_horizon: int, dropout: float = 0.2):
        """
        Parameters
        ----------
        input_size : int
            Number of input features (1 for univariate, >1 for multivariate)
        hidden_size : int
            LSTM hidden size
        num_layers : int
            Number of LSTM layers
        forecast_horizon : int
            Number of steps to forecast
        dropout : float
            Dropout rate
        """
        super(LSTMForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.forecast_horizon = forecast_horizon
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # Output layer
        self.fc = nn.Linear(hidden_size, forecast_horizon)
        
    def forward(self, x):
        """
        Parameters
        ----------
        x : torch.Tensor
            Shape: (batch_size, lookback, input_size)
        
        Returns
        -------
        torch.Tensor
            Shape: (batch_size, forecast_horizon)
        """
        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Use last hidden state for prediction
        last_hidden = h_n[-1]  # Shape: (batch_size, hidden_size)
        
        # Predict forecast horizon
        output = self.fc(last_hidden)  # Shape: (batch_size, forecast_horizon)
        
        return output


In [56]:
class MultiProductForecaster:
    """Forecaster for multiple products with proper train/val/test split."""
    
    def __init__(self, train_ratio: float = 0.6, val_ratio: float = 0.2,
                 lookback_days: int = 60,  # Fixed lookback window
                 hidden_size: int = 128, num_layers: int = 2):
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = 1.0 - train_ratio - val_ratio
        self.lookback_days = lookback_days  # Fixed lookback (not dynamic)
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.models = {}
        self.scalers = {}
        self.forecast_horizons = {}
        
    def calculate_splits(self, total_length: int) -> Tuple[int, int, int]:
        """Calculate train/val/test split indices."""
        train_end = int(total_length * self.train_ratio)
        val_end = int(total_length * (self.train_ratio + self.val_ratio))
        test_length = total_length - val_end  # Forecast horizon
        return train_end, val_end, test_length
        
    def prepare_data(self, df: pd.DataFrame, store_id: int, item_id: int,
                    include_features: List[str] = None):
        """Prepare train/val/test datasets for a single product."""
        
        # Filter to store-item
        product_data = df[
            (df['store_id'] == store_id) & (df['item_id'] == item_id)
        ].sort_values('date').reset_index(drop=True)
        
        total_length = len(product_data)
        train_end, val_end, test_length = self.calculate_splits(total_length)
        forecast_horizon = test_length
        
        # Check if we have enough data
        min_required = self.lookback_days + forecast_horizon + 10  # Some margin
        if train_end < self.lookback_days + 10:
            print(f"Skipping item {item_id}: insufficient training data (need >{self.lookback_days + 10}, have {train_end})")
            return None, None, None, None, None
        
        # Store forecast horizon for this product
        key = (store_id, item_id)
        self.forecast_horizons[key] = forecast_horizon
        
        # Scale the values
        scaler = StandardScaler()
        product_data['value_scaled'] = scaler.fit_transform(product_data[['value']])
        
        # Store scaler
        self.scalers[key] = scaler
        
        # Create temporary dataframe with scaled values
        temp_df = product_data.copy()
        temp_df['value'] = temp_df['value_scaled']
        
        # Split into train/val/test
        train_df = temp_df.iloc[:train_end]
        val_df = temp_df.iloc[:val_end]  # Val includes train data for sequence creation
        test_df = temp_df  # Test includes all data for sequence creation
        
        # Create datasets
        train_dataset = TimeSeriesDataset(
            train_df, self.lookback_days, forecast_horizon,
            store_id, item_id, include_features
        )
        
        val_dataset = TimeSeriesDataset(
            val_df, self.lookback_days, forecast_horizon,
            store_id, item_id, include_features
        )
        
        test_dataset = TimeSeriesDataset(
            test_df, self.lookback_days, forecast_horizon,
            store_id, item_id, include_features
        )
        
        print(f"  Total: {total_length} | Lookback: {self.lookback_days} | Train: {train_end} | Val: {val_end-train_end} | Test: {test_length} | Train sequences: {len(train_dataset)}")
        
        return train_dataset, val_dataset, test_dataset, scaler, forecast_horizon
    
    def train_product(self, store_id: int, item_id: int, 
                     train_dataset: Dataset, val_dataset: Dataset,
                     forecast_horizon: int, num_epochs: int = 50, 
                     learning_rate: float = 0.001, batch_size: int = 32,
                     early_stopping_patience: int = 10):
        """Train LSTM model for a single product with validation."""
        
        if train_dataset is None or len(train_dataset) == 0:
            return None, None
        
        # Create data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) if val_dataset and len(val_dataset) > 0 else None
        
        # Initialize model
        input_size = train_dataset.X.shape[2]
        model = LSTMForecaster(
            input_size=input_size,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            forecast_horizon=forecast_horizon
        ).to(DEVICE)
        
        # Loss and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
        # Training loop with validation
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        
        for epoch in range(num_epochs):
            # Training phase
            model.train()
            total_train_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                
                # Forward pass
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_train_loss += loss.item()
            
            avg_train_loss = total_train_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation phase
            if val_loader:
                model.eval()
                total_val_loss = 0
                with torch.no_grad():
                    for X_batch, y_batch in val_loader:
                        X_batch = X_batch.to(DEVICE)
                        y_batch = y_batch.to(DEVICE)
                        outputs = model(X_batch)
                        loss = criterion(outputs, y_batch)
                        total_val_loss += loss.item()
                
                avg_val_loss = total_val_loss / len(val_loader)
                val_losses.append(avg_val_loss)
                
                # Early stopping check
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    patience_counter = 0
                    best_model_state = model.state_dict().copy()
                else:
                    patience_counter += 1
                
                if (epoch + 1) % 10 == 0:
                    print(f"  Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
                
                # Early stopping
                if patience_counter >= early_stopping_patience:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break
            else:
                if (epoch + 1) % 10 == 0:
                    print(f"  Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.6f}")
        
        # Load best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        
        # Store model
        key = (store_id, item_id)
        self.models[key] = model
        
        return model, {'train_losses': train_losses, 'val_losses': val_losses}
    
    def predict(self, store_id: int, item_id: int, 
                recent_data: pd.DataFrame) -> np.ndarray:
        """
        Generate forecast for a product (size = test set size = 20% of data).
        
        Parameters
        ----------
        store_id, item_id : int
            Product identifier
        recent_data : pd.DataFrame
            Recent data (at least lookback days) for the product
        
        Returns
        -------
        np.ndarray
            Forecast (unscaled) with length = forecast_horizon
        """
        key = (store_id, item_id)
        
        if key not in self.models:
            print(f"No model found for store {store_id}, item {item_id}")
            return None
        
        model = self.models[key]
        scaler = self.scalers[key]
        lookback = self.lookback_windows[key]
        
        # Prepare input
        product_data = recent_data[
            (recent_data['store_id'] == store_id) & 
            (recent_data['item_id'] == item_id)
        ].sort_values('date').tail(lookback)
        
        if len(product_data) < lookback:
            print(f"Insufficient recent data for item {item_id}")
            return None
        
        # Scale values
        values_scaled = scaler.transform(product_data[['value']])
        
        # Create input tensor
        X = torch.FloatTensor(values_scaled).unsqueeze(0).to(DEVICE)  # Shape: (1, lookback, 1)
        
        # Predict
        model.eval()
        with torch.no_grad():
            forecast_scaled = model(X).cpu().numpy()[0]
        
        # Inverse transform
        forecast = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()
        
        return forecast
    
    def train_all_products(self, df: pd.DataFrame, store_id: int = None,
                          num_epochs: int = 50, learning_rate: float = 0.001):
        """Train models for all products in the dataset."""
        
        if store_id is not None:
            product_pairs = [(store_id, item_id) for item_id in df[df['store_id'] == store_id]['item_id'].unique()]
        else:
            product_pairs = df.groupby(['store_id', 'item_id']).size().index.tolist()
        
        results = {}
        
        print(f"\n{'='*80}")
        print(f"Training LSTM models for {len(product_pairs)} products")
        print(f"Lookback: {self.lookback_days} days | Train {self.train_ratio:.0%} | Val {self.val_ratio:.0%} | Test {self.test_ratio:.0%}")
        print(f"{'='*80}\n")
        
        for i, (s_id, i_id) in enumerate(product_pairs):
            print(f"[{i+1}/{len(product_pairs)}] Training model for Store {s_id}, Item {i_id}")
            
            # Prepare data
            result_tuple = self.prepare_data(df, s_id, i_id)
            
            if result_tuple[0] is None:
                continue
            
            train_ds, val_ds, test_ds, scaler, forecast_horizon = result_tuple
            
            # Train model with validation
            model, losses = self.train_product(
                s_id, i_id, train_ds, val_ds, forecast_horizon,
                num_epochs=num_epochs, 
                learning_rate=learning_rate
            )
            
            results[(s_id, i_id)] = {
                'model': model,
                'losses': losses,
                'lookback': self.lookback_days,
                'train_size': len(train_ds) if train_ds else 0,
                'val_size': len(val_ds) if val_ds else 0,
                'test_size': len(test_ds) if test_ds else 0,
                'forecast_horizon': forecast_horizon
            }
        
        return results



In [57]:
# Initialize forecaster with 60/20/20 split
forecaster = MultiProductForecaster(
        train_ratio=TRAIN_RATIO,  # 60% (also lookback)
        val_ratio=VAL_RATIO,      # 20%
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS
    )
    
    # Train models for all products in a specific store
selected_store = 6269
    
results = forecaster.train_all_products(
        df_selected, 
        store_id=selected_store,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE
    )
    
print(f"\n{'='*80}")
print(f"Training complete! Trained {len(results)} models.")
print(f"{'='*80}")
    
# Show lookback and forecast horizons
print("\nModel configurations per product:")
for (s_id, i_id), result_info in list(results.items())[:5]:
    print(f"  Store {s_id}, Item {i_id}: Lookback={result_info['lookback']} days, Forecast={result_info['forecast_horizon']} days")
    
# Generate forecasts for all products
forecasts = {}
for (s_id, i_id) in results.keys():
    forecast = forecaster.predict(s_id, i_id, df_selected)
    if forecast is not None:
        forecasts[(s_id, i_id)] = forecast
        print(f"Store {s_id}, Item {i_id}: Forecast generated ({len(forecast)} days, mean: {forecast.mean():.2f})")


Training LSTM models for 996 products
Lookback: 60 days | Train 60% | Val 20% | Test 20%

[1/996] Training model for Store 6269, Item 27
  Total: 761 | Lookback: 60 | Train: 456 | Val: 152 | Test: 153 | Train sequences: 244
  Epoch [10/50], Train Loss: 1.002604, Val Loss: 1.067647
  Early stopping at epoch 11
[2/996] Training model for Store 6269, Item 952568
  Total: 761 | Lookback: 60 | Train: 456 | Val: 152 | Test: 153 | Train sequences: 244
  Epoch [10/50], Train Loss: 1.007865, Val Loss: 1.047042
  Epoch [20/50], Train Loss: 0.932269, Val Loss: 1.036201
  Epoch [30/50], Train Loss: 0.809506, Val Loss: 0.988090
  Epoch [40/50], Train Loss: 0.770236, Val Loss: 0.966811
  Epoch [50/50], Train Loss: 0.711716, Val Loss: 0.951669
[3/996] Training model for Store 6269, Item 809
  Total: 761 | Lookback: 60 | Train: 456 | Val: 152 | Test: 153 | Train sequences: 244
  Epoch [10/50], Train Loss: 0.991603, Val Loss: 1.082610
  Early stopping at epoch 11
[4/996] Training model for Store 6269,

KeyboardInterrupt: 

In [ ]:
# Debug: Check why models weren't trained
print("Checking training results...")
print(f"Results dictionary has {len(results)} entries")

# Check if any models actually exist
models_trained = sum(1 for v in results.values() if v.get('model') is not None)
print(f"Models successfully trained: {models_trained}")

# Examine one product to see what went wrong
if len(results) > 0:
    first_key = list(results.keys())[0]
    first_result = results[first_key]
    print(f"\nFirst result for {first_key}:")
    for k, v in first_result.items():
        if k != 'model':
            print(f"  {k}: {v}")

# Check data availability for a specific product
sample_store = 6269
sample_item = 27
sample_data = df_selected[
    (df_selected['store_id'] == sample_store) & 
    (df_selected['item_id'] == sample_item)
]
print(f"\nSample product {sample_item} data:")
print(f"  Records: {len(sample_data)}")
if len(sample_data) > 0:
    print(f"  Date range: {sample_data['date'].min()} to {sample_data['date'].max()}")
    lookback_needed = int(len(sample_data) * 0.6)
    forecast_needed = int(len(sample_data) * 0.2)
    total_needed = lookback_needed + forecast_needed
    print(f"  Lookback needed (60%): {lookback_needed}")
    print(f"  Forecast needed (20%): {forecast_needed}")
    print(f"  Total needed: {total_needed}")
    print(f"  Sufficient data: {len(sample_data) >= total_needed}")

Checking training results...
Results dictionary has 996 entries
Models successfully trained: 0

First result for (6269, np.int64(27)):
  losses: None
  lookback: 456
  train_size: 0
  val_size: 0
  test_size: 153
  forecast_horizon: 153

Sample product 27 data:
  Records: 761
  Date range: 2021-01-23 00:00:00 to 2023-02-22 00:00:00
  Lookback needed (60%): 456
  Forecast needed (20%): 152
  Total needed: 608
  Sufficient data: True
